# 01 · Define & Explore — the MPNN trade-off + the design/recapitulation loop

**Standard slot:** *define & explore.* **For Project 02 this means:** pin down the
recovery ↔ foldability ↔ expressibility trade-off and the metrics that measure it, then stand up
`mpnn_tools` plus a recapitulation step and run them on **one** backbone as your "hello-world" (D0):
design sequences, compute sequence recovery, and a (mock) recapitulation scRMSD.

Run `00_setup.ipynb` first in this session.

## The metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| Sequence recovery | 0–1 | fraction of positions matching a native/reference | design *quality* (high recovery can just mean low diversity) |
| Recapitulation scRMSD | Å | designed→predicted-vs-input-backbone Cα-RMSD (<2 Å self-consistent) | that the protein folds, is stable, or expresses |
| pLDDT (mean) | 0–100 | local confidence of the recapitulation | thermostability / expression |
| Per-position entropy | bits | sequence *diversity* across samples | correctness |
| Net charge (pH 7.4) | ± | a solubility *proxy* (charge extremes hurt solubility) | a measured solubility |
| Hydrophobic-patch fraction | 0–1 | aggregation-risk *proxy* (exposed hydrophobic runs) | a measured aggregation rate |
| `camsol_like` | unitless | a CamSol-**style** heuristic (higher ≈ more soluble) | **real CamSol**; not an expression result |

Write your own one-paragraph definitions in `D0` — including the "does not mean" column, which is
where wasted synthesis budget comes from (e.g., treating a proxy score as an expression prediction).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Stand up `mpnn_tools`

`scripts/mpnn_tools.py` exposes `run_mpnn(backbone, temperature, noise, n_seqs, tool)` plus the
metric helpers. The real backend shells out to ProteinMPNN (see the TODO in `_real_mpnn`); a
deterministic **mock** backend lets you build and test the sweep, recovery, entropy, and solubility
logic first. **Never report mock sequences or proxy scores as real results.**

In [ ]:
from mpnn_tools import (run_mpnn, sequence_recovery, shannon_entropy,
                        net_charge, hydrophobic_fraction, camsol_like)

# Design 8 sequences for ONE backbone with the mock backend (no GPU, no MPNN install).
designs = run_mpnn("demo_backbone_01", temperature=0.2, noise=0.1, n_seqs=8, tool="mock")
print(f"designed {len(designs)} sequences for backbone {designs[0].backbone!r}")
print("first sequence:", designs[0].sequence)

## Hello-world: recovery + a mock recapitulation for one backbone

Sequence recovery needs a *native* reference. For a de novo backbone there is no single native —
use the mock backend's own consensus as a stand-in here so the plumbing runs; with the real backend
you compute recovery vs the backbone's reference sequence (natural references) or report inter-design
recovery (de novo). Then we *recapitulate*: a real run predicts each sequence's structure (ESMFold/AF2)
and measures Cα-RMSD back to the input backbone. Here a small **mock predict** stands in so the loop
executes anywhere.

In [ ]:
import hashlib

# A stand-in "native" so recovery is meaningful in the dry run: re-derive the mock consensus.
# (With the real backend, recovery is vs the backbone's reference sequence — see MANUAL §2.)
ref = run_mpnn("demo_backbone_01", temperature=0.01, noise=0.0, n_seqs=1, tool="mock")[0].sequence

def mock_recapitulate(sequence):
    """Deterministic stand-in for ESMFold/AF2 recapitulation. NOT a real prediction.
    Returns (scrmsd_A, plddt). Replace with a real predict() on Colab (see scripts/predict.py
    pattern in Project 01)."""
    h = int(hashlib.sha256(sequence.encode()).hexdigest(), 16)
    scrmsd = round(0.8 + (h % 350) / 100.0, 2)   # 0.8–4.3 Å, synthetic
    plddt = round(60 + (h % 40), 1)              # 60–99, synthetic
    return scrmsd, plddt

for d in designs:
    d.recovery = sequence_recovery(ref, d.sequence)
    d.scrmsd, d.plddt = mock_recapitulate(d.sequence)

print("EXAMPLE_DATA (mock backend) — one backbone, 8 sequences:")
for d in designs:
    print(f"  seq {d.seq_index}: recovery={d.recovery:.2f}  scrmsd={d.scrmsd} A  "
          f"plddt={d.plddt}  net_charge={d.net_charge}  hp_frac={d.hydrophobic_fraction}  "
          f"camsol_like={d.camsol_like}")

### Diversity at one glance

Per-position Shannon entropy across the 8 sequences — the diversity axis you will sweep against
temperature. Near-zero means MPNN is confident (low temperature); higher means it samples broadly.

In [ ]:
ent = shannon_entropy([d.sequence for d in designs])
print(f"mean per-position entropy = {ent['mean']:.3f} bits over {ent['length']} positions")
print("Try re-running run_mpnn(...) with temperature=0.1 vs 0.5 and watch entropy move.")

### Switch to the real backend (on Colab)

Once you have cloned ProteinMPNN (pin the commit) and installed ESMFold via `00_setup`'s
`install_esmfold()`, change `tool="mock"` to `tool="proteinmpnn"` in `run_mpnn(...)` and replace
`mock_recapitulate` with a real `predict(sequence, tool="esmfold")` → scRMSD. Record runtime +
versions + the pinned commit in `LOG.md`. **MPNN is seconds; the predictor is your compute cost.**

In [ ]:
# Uncomment on Colab after wiring up ProteinMPNN + ESMFold:
# real = run_mpnn("data/inputs/backbone_001.pdb", temperature=0.2, noise=0.1, n_seqs=8,
#                 tool="proteinmpnn")
# from predict import predict   # Project 01-style wrapper; compute Ca-RMSD vs the input backbone
# ...
print("Ready — flip tool='mock' to 'proteinmpnn' and wire in a real predict() on Colab.")

## D0 checklist
- [ ] One-paragraph definition of each metric **with** its "does not mean" note (esp. proxy ≠ expression).
- [ ] One backbone designed; sequence recovery + a (mock, then real) recapitulation scRMSD printed.
- [ ] Per-position entropy printed; you have watched it move with temperature.
- [ ] `LOG.md` entry: tool version + pinned ProteinMPNN commit, GPU, runtime, seed.

**Next:** `02_generate.ipynb` — the systematic sweep over the grid.